# Musanga — NB06 V4.002 | Quick point-cloud added-value experiment

**Goal:** investigate whether LAS/LAZ-derived 3D canopy structure provides information beyond a co-registered DSM, on the **original frozen spatial held-out reference units**. Keep this exploratory experiment separate from the publishable Option B U-Net until its point-cloud quality and alignment are confirmed.

- **Input**: `Final/Data/LAS` (searches `Data/LAS` recursively), including independent `part_1` … `part_6` files. We **do not concatenate or merge** the large files.
- **Site mapping**: `Carlier2020_Aug20*` files are explicitly **Yangambi**. Baego, MOS, Yangambi, Yoko are otherwise inferred from filenames and spatially checked against original reference units. Unmatched files remain unassigned (not mislabeled).
- **Dataset**: Original exhaustive *field plots* and their Musanga crown polygons; spatial split = NB01 fixed field-plot folds (0,1,2). Circles are excluded from this first quick test to keep label interpretation transparent. Crown label = `Class_ID == 1`.
- **Features**: 2-metre XY grid cells containing sampled 3D points: returns per-cell point density, vertical 10/50/90 percentiles, vertical range and spread. **These are relative vertical-structure signals**, not necessarily canopy height above ground; LAS z may represent ellipsoidal/absolute elevation or dense-matching surfaces.
- **Comparison**: DSM-only *matched cell* texture/elevation predictors versus DSM + LAS vertical structure. A simple RF classifier serves as a *screening tool*, not a U-Net equivalent. Per-cell predictions aggregate to *plot-level cover* and F1, held out by the frozen plot fold. No pixel-level interplot leakage.
- **Budget**: small chunked sample from each LAS (default at most 1,000,000 source points/file), no full-cloud scanning, no writing merged LAS. Header-only inventory takes seconds. A limited sample may cover only an early spatial portion of a LAS file: **a missing plot is not evidence of no cloud coverage**. Results with <2 valid sites/folds or class imbalance are diagnostic only.
- **Status**: exploratory benchmark; do not claim a causal point-cloud benefit or publishable validation from sparse partial LAS reading. Use this to decide if full spatial-indexed point-cloud features are worth engineering.
  
**Run:** `00 → 01 → 02 → 03 → 04`. Use a **CPU** runtime. If needed, start with `MAX_POINTS_FILE = 200_000`, then increase to 1,000,000. Every completed file has a provenance-checked resumable cache.

**V4.002 corrections:** The names `YBForgeo` and `YBJoseph` are Yangambi, together with `Carlier2020_Aug20`. Use stratified file-offset samples rather than only reading the first million points, avoiding a strong bias from spatial point ordering. The cloud is likely photogrammetric dense matching: avoid interpreting Z percentiles as direct undercanopy LiDAR penetration. Repeat the inventory and extraction in the new notebook; previously extracted quick-test caches remain untouched.


## 00 — CPU setup, frozen reference, and file paths

In [ ]:
%pip -q install "laspy[lazrs]" geopandas shapely rasterio scipy scikit-learn
from pathlib import Path
from collections import defaultdict
import os,re,json,hashlib,gc
import numpy as np,pandas as pd,geopandas as gpd,rasterio
import laspy
from shapely.geometry import box
from shapely.strtree import STRtree
from shapely import contains_xy
from scipy.spatial import cKDTree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score,f1_score,roc_auc_score
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final')
DATA=ROOT/'Data'
LAS_ROOT=DATA/'LAS'
REF=ROOT/'Outputs/v4/01_reference_field78_combined'
OUT=ROOT/'Outputs/v4/06_pointcloud_quick_test_v4_002'
OUT.mkdir(parents=True,exist_ok=True)
assert LAS_ROOT.is_dir(),('LAS folder not found',LAS_ROOT)
CONTRACT=json.loads((REF/'01_field78_reference_contract.json').read_text())
NB00=json.loads(Path(CONTRACT['input_manifest']).read_text())
PLOTS=gpd.read_file(NB00['field_support_qa']).to_crs(32635)
FOLDS=pd.read_csv(CONTRACT['folds_csv'])
PLOTS=PLOTS.merge(FOLDS[['source_id','site_key','cv_fold']],left_on='Plot_name',
                  right_on='source_id',validate='one_to_one')
CROWNS=gpd.read_file(NB00['exhaustive_plot_crowns']).to_crs(32635)
CROWNS=CROWNS.loc[pd.to_numeric(CROWNS.Class_ID,errors='coerce').eq(1)].copy()
assert len(PLOTS)==78 and set(PLOTS.cv_fold)=={0,1,2}
PLOT_GEOMS=list(PLOTS.geometry)
PLOT_TREE=STRtree(PLOT_GEOMS)
CROWN_GEOMS=list(CROWNS.geometry)
CROWN_TREE=STRtree(CROWN_GEOMS)
print('Study plots by site/fold')
display(PLOTS.groupby(['site_key','cv_fold']).size().unstack(fill_value=0))
print('LAS ROOT',LAS_ROOT)


## 01 — Fast header-only LAS/LAZ inventory; recognize split parts and Yangambi aliases

In [ ]:
def site_from_name(s):
    s=s.lower()
    if any(z in s for z in ('carlier2020_aug20','yangambi','ybforgeo','ybjoseph')):return 'Yangambi'
    if 'baego' in s:return 'Baego'
    if re.search(r'(^|[^a-z])mos([^a-z]|$)',s):return 'MOS'
    if 'yoko' in s:return 'Yoko'
    return 'UNASSIGNED'
FILES=sorted(p for p in LAS_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in ('.las','.laz'))
assert FILES,('No LAS or LAZ files detected',LAS_ROOT)
HEADERS=[]
for p in FILES:
    try:
        with laspy.open(p) as reader:
            h=reader.header
            crs=h.parse_crs()
            # Do not silently assert a CRS when absent.
            epsg=crs.to_epsg() if crs is not None else None
            bb=[float(v) for v in h.mins]+[float(v) for v in h.maxs]
            part=re.search(r'part[_-]?(\d+)',p.stem.lower())
            HEADERS.append(dict(path=str(p),name=p.name,site=site_from_name(str(p.relative_to(LAS_ROOT))),
                part=int(part.group(1)) if part else None,
                points=int(h.point_count),bytes=p.stat().st_size,
                epsg=epsg,min_x=bb[0],min_y=bb[1],min_z=bb[2],
                max_x=bb[3],max_y=bb[4],max_z=bb[5],error=''))
    except Exception as exc:
        HEADERS.append(dict(path=str(p),name=p.name,site='ERROR',part=None,
            points=0,bytes=p.stat().st_size,epsg=None,
            min_x=np.nan,min_y=np.nan,min_z=np.nan,
            max_x=np.nan,max_y=np.nan,max_z=np.nan,error=repr(exc)))
LAS_INDEX=pd.DataFrame(HEADERS)
LAS_INDEX.to_csv(OUT/'06_las_header_inventory.csv',index=False)
display(LAS_INDEX[['site','name','part','points','epsg','bytes','error']])
print('Total files',len(FILES),'| header readable',sum(LAS_INDEX.error.eq('')))
print('Mapped site file counts:')
display(LAS_INDEX.groupby('site').agg(files=('name','size'),
    points=('points','sum'),gb=('bytes',lambda v:round(v.sum()/1e9,2))))


## 02 — Budgeted chunk streaming (no LAS merge)

A header bounding-box overlap is needed before processing a file. Read in chunks, retain only points falling within one of the original field plots, and bin the hits at **2 m**. If a LAS has parts, each part is independently streamed and aggregated into a single site's grid. Cap total file reads to keep costs bounded. Spatially ordered LAS files can under-sample some plots: inspect the per-plot support table before interpreting any model comparison.

In [ ]:
# Budget can be raised later. No attempt to scan gigabytes of point clouds at once.
MAX_POINTS_FILE=1_000_000
N_STRATA=8  # distributed intervals along source file; LAS is often spatially ordered
LAS_CHUNK=250_000
GRID_M=2.0
MAX_POINTS_PER_CELL=200 # deterministic first-N cap per grid cell
CELL_OUT=OUT/'02_sampled_cells'
CELL_OUT.mkdir(parents=True,exist_ok=True)
from rasterio.warp import transform_bounds
from pyproj import CRS,Transformer
def hash_input(p):
    st=p.stat()
    return hashlib.sha256(json.dumps(dict(file=str(p),bytes=st.st_size,
      mtime=st.st_mtime_ns,budget=MAX_POINTS_FILE,
      chunk=LAS_CHUNK,strata=N_STRATA,grid_m=GRID_M,max_per_cell=MAX_POINTS_PER_CELL,
      recipe='NB06_V4_002_stratified_seek_2m_cell_v1'),sort_keys=True).encode()).hexdigest()
def point_group(name,epsg):
    # CRS must be declared: spatial diagnostics cannot be trusted otherwise.
    if pd.isna(epsg) or int(epsg)!=32635:return None
    if name not in ('Baego','MOS','Yangambi','Yoko'):return None
    return name
def sample_las_file(record):
    p=Path(record.path)
    key=hash_input(p)
    dest=CELL_OUT/f'{key[:26]}.npz'
    if dest.exists():
        with np.load(dest,allow_pickle=False) as z:
            if str(z['signature'].item())==key:
                return dest
    candidates=PLOTS.loc[PLOTS.site_key.eq(record.site)]
    envelope=box(float(record.min_x),float(record.min_y),float(record.max_x),float(record.max_y))
    chosen=[(int(i),g) for i,g in zip(candidates.index,candidates.geometry)
         if g.intersects(envelope)]
    if not chosen:
        print('NO PLOT BBOX overlap:',p.name,flush=True);return None
    # Grid cells belong to one original plot; never leak across folds.
    buckets=defaultdict(list)
    seen_counts=defaultdict(int)
    processed=0
    with laspy.open(p) as reader:
        n_total=int(reader.header.point_count)
        n_each=min(LAS_CHUNK,max(1,MAX_POINTS_FILE//N_STRATA))
        offsets=np.linspace(0,max(0,n_total-n_each),N_STRATA,dtype=np.int64)
        # Each stratum samples one bounded interval; avoids first-file-only selection.
        for start in np.unique(offsets):
            reader.seek(int(start))
            cloud=reader.read_points(int(min(n_each,n_total-int(start))))
            x=np.asarray(cloud.x);y=np.asarray(cloud.y);z=np.asarray(cloud.z)
            processed+=len(x)
            valid=np.isfinite(x)&np.isfinite(y)&np.isfinite(z)
            for plot_idx,geom in chosen:
                x0,y0,x1,y1=geom.bounds
                pre=valid&(x>=x0)&(x<=x1)&(y>=y0)&(y<=y1)
                if not pre.any():continue
                ix=np.flatnonzero(pre)
                # Polygon predicate in vectorized form, only inside a small bbox.
                inside=contains_xy(geom,x[ix],y[ix])
                ix=ix[inside]
                if not len(ix):continue
                xc=np.floor(x[ix]/GRID_M).astype('int64')
                yc=np.floor(y[ix]/GRID_M).astype('int64')
                for i,xx,yy in zip(ix,xc,yc):
                    key2=(plot_idx,int(xx),int(yy))
                    if seen_counts[key2]>=MAX_POINTS_PER_CELL:continue
                    seen_counts[key2]+=1
                    buckets[key2].append(float(z[i]))
            if processed>=MAX_POINTS_FILE:break
    records=[]
    for (idx,xx,yy),zz in buckets.items():
        if len(zz)<4:continue
        a=np.asarray(zz)
        records.append(dict(plot_index=idx,site=record.site,
            plot=str(PLOTS.loc[idx,'Plot_name']),
            fold=int(PLOTS.loc[idx,'cv_fold']),
            gx=xx,gy=yy,n=len(a),
            z10=float(np.percentile(a,10)),z50=float(np.median(a)),
            z90=float(np.percentile(a,90)),z_sd=float(np.std(a))))
    d=pd.DataFrame(records,columns=['plot_index','site','plot','fold',
         'gx','gy','n','z10','z50','z90','z_sd'])
    tmp=dest.with_suffix('.partial.npz')
    np.savez_compressed(tmp,signature=np.array(key),
        **{col:d[col].to_numpy() for col in d.columns})
    os.replace(tmp,dest)
    print('CLOUD SAMPLED',p.name,'points read',processed,'2m cells',len(d),flush=True)
    return dest
CACHE_FILES=[]
for r in LAS_INDEX.itertuples(index=False):
    if r.error or point_group(r.site,r.epsg) is None:
        print('SKIP (bad header, unassigned site, or missing/not-UTM35 CRS)',r.name,r.epsg)
        continue
    result=sample_las_file(r)
    if result:CACHE_FILES.append(str(result))
assert CACHE_FILES,'No sample point-cloud grids; inspect LAS header EPSG and spatial overlap'
(OUT/'06_sampled_cache_paths.json').write_text(json.dumps(CACHE_FILES,indent=2))
print('Cached file samples',len(CACHE_FILES),'of',len(FILES))


## 03 — Merge *sampled per-cell statistics*, label by crowns and match the DSM

This aggregates sampled cells across LAS split parts **without merging LAS files**. Since the file-level record contains only summary statistics, the combination uses weighted means and upper/lower percentile summaries as **rough approximations**; duplicate spatial regions are not independent and the full-cloud percentiles are not recoverable from these summaries. A next version should compute exact point-level accumulators for publication. Here we compare simple DSM elevation/local texture and density/vertical range at exactly the same labelled grid cells.

In [ ]:
from rasterio.windows import Window
files=json.loads((OUT/'06_sampled_cache_paths.json').read_text())
parts=[]
for f in files:
    with np.load(f,allow_pickle=False) as z:
        d=pd.DataFrame({k:z[k] for k in z.files if k!='signature'})
    if len(d):parts.append(d)
assert parts,'No plot-cell samples in the limited read budget'
raw=pd.concat(parts,ignore_index=True)
raw['plot_index']=raw.plot_index.astype(int)
raw['gx']=raw.gx.astype('int64');raw['gy']=raw.gy.astype('int64')
# Weighted combination of descriptive summaries. This is an exploratory proxy only.
raw['weighted_z10']=raw.z10*raw.n
raw['weighted_z50']=raw.z50*raw.n
raw['weighted_z90']=raw.z90*raw.n
raw['weighted_zsd']=raw.z_sd*raw.n
CELL=raw.groupby(['plot_index','site','plot','fold','gx','gy'],as_index=False).agg(
    n=('n','sum'),z10_sum=('weighted_z10','sum'),
    z50_sum=('weighted_z50','sum'),z90_sum=('weighted_z90','sum'),
    zsd_sum=('weighted_zsd','sum'))
for b in ('z10','z50','z90'):
    CELL[b]=CELL[b+'_sum']/CELL.n
CELL['z_sd']=CELL.zsd_sum/CELL.n
CELL['vertical_80']=CELL.z90-CELL.z10
CELL['x']=(CELL.gx+.5)*GRID_M
CELL['y']=(CELL.gy+.5)*GRID_M
# Only an original plot site and polygon support count as supervised.
from shapely import contains_xy
accept=np.zeros(len(CELL),bool)
for idx,subset in CELL.groupby('plot_index'):
    ix=subset.index.to_numpy()
    accept[ix]=contains_xy(PLOTS.loc[int(idx),'geometry'],
              CELL.loc[ix,'x'].to_numpy(),CELL.loc[ix,'y'].to_numpy())
CELL=CELL.loc[accept].copy().reset_index(drop=True)
# Musanga truth only from original exhaustive plot crown polygons.
tree=STRtree(CROWN_GEOMS)
truth=[]
for r in CELL.itertuples(index=False):
    p=box(r.x-.01,r.y-.01,r.x+.01,r.y+.01)
    hits=tree.query(p,predicate='intersects')
    truth.append(any(CROWN_GEOMS[int(k)].covers(p.centroid) for k in hits))
CELL['musanga']=np.asarray(truth,'uint8')
# NB00 original DSM pairing; use the paired dataset with matching site and RGB mosaic.
PAIRS=pd.read_csv(ROOT/'Outputs/v4/00_input_audit/00_rgb_dsm_pair_audit.csv')
DSM_BY_MOSAIC={Path(r.rgb_path).name:Path(r.dsm_path) for r in PAIRS.itertuples(index=False)}
CV=pd.read_csv(CONTRACT['cv_manifest'])
SITE_MOSAICS={site:sorted(CV.loc[CV.site.eq(site),'mosaic'].unique())
              for site in PLOTS.site_key.unique()}
from contextlib import ExitStack
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
CELL['dsm_z']=np.nan
CELL['dsm_local_sd']=np.nan
CELL['dsm_local_range']=np.nan
for site,indices in CELL.groupby('site').groups.items():
    # Each plot-cell can intersect multiple RGB mosaics. Pick first spatially valid DSM.
    for mosaic in SITE_MOSAICS.get(site,[]):
        if mosaic not in DSM_BY_MOSAIC:continue
        remaining=CELL.loc[list(indices)]
        remaining=remaining[remaining.dsm_z.isna()]
        if remaining.empty:break
        with rasterio.open(DSM_BY_MOSAIC[mosaic]) as ds:
            if ds.crs is None or ds.crs.to_epsg()!=32635:continue
            for idx,row in remaining.iterrows():
                rr,cc=ds.index(float(row.x),float(row.y))
                if rr<3 or cc<3 or rr>=ds.height-3 or cc>=ds.width-3:continue
                win=Window(cc-3,rr-3,7,7)
                a=ds.read(1,window=win,masked=True)
                z=a.compressed()
                if len(z)<25:continue
                vals=np.asarray(z,dtype=np.float32)
                CELL.at[idx,'dsm_z']=float(np.median(vals))
                CELL.at[idx,'dsm_local_sd']=float(np.std(vals))
                CELL.at[idx,'dsm_local_range']=float(np.percentile(vals,90)-np.percentile(vals,10))
CELL=CELL.dropna(subset=['dsm_z','dsm_local_sd','dsm_local_range']).copy()
CELL['log_n']=np.log1p(CELL.n.clip(0,500))
assert not CELL.empty,'No DSM-matched cloud cells'
CELL.to_csv(OUT/'06_exploratory_las_dsm_matched_cells.csv',index=False)
display(CELL.groupby(['site','fold']).agg(
    cells=('musanga','size'),positive=('musanga','sum'),
    original_plots=('plot','nunique'),median_cloud_points=('n','median')))
print('Cells are sparse samples from partial LAS reads, NOT full-plot coverage.')


## 04 — Fixed-fold added-value screen (DSM vs DSM + LAS)

**Interpret only if each held-out fold has both classes** and sufficient independent plots. No site- or fold-specific tuning. Compute held-out balanced accuracy, F1 and absolute cover bias on the **same sampled cells**; plot-level percentage is a *sparse-sample proxy*, not true full-area fractional cover. Do not extrapolate to whole plots without a complete point-cloud map.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score,f1_score
BASE=['dsm_z','dsm_local_sd','dsm_local_range']
PLUS=BASE+['log_n','vertical_80','z_sd']
evaluations=[];predictions=[]
for fold in (0,1,2):
    tr=CELL[CELL.fold.ne(fold)].copy()
    te=CELL[CELL.fold.eq(fold)].copy()
    if tr.musanga.nunique()<2 or te.musanga.nunique()<2 or len(te)<30:
        print('FOLD',fold,'not enough balanced spatially held-out sampled cells; skipping')
        continue
    for modelname,features in [('DSM',BASE),('DSM+LAS',PLUS)]:
        clf=RandomForestClassifier(n_estimators=120,min_samples_leaf=10,
            max_features='sqrt',class_weight='balanced_subsample',
            random_state=20261117,n_jobs=-1)
        clf.fit(tr[features],tr.musanga)
        prob=clf.predict_proba(te[features])[:,list(clf.classes_).index(1)]
        pred=(prob>=.5).astype(int)
        evaluations.append(dict(fold=fold,model=modelname,cells=len(te),
            plots=te['plot'].nunique(),
            balanced_accuracy=balanced_accuracy_score(te.musanga,pred),
            f1=f1_score(te.musanga,pred,zero_division=0)))
        for k,q in te.reset_index(drop=True).iterrows():
            predictions.append(dict(fold=fold,model=modelname,site=q.site,
                plot=q['plot'],y=int(q.musanga),pred=int(pred[k]),prob=float(prob[k])))
RES=pd.DataFrame(evaluations)
if RES.empty:
    print('Insufficient class/fold support in quick sampled LAS test.')
else:
    PRED=pd.DataFrame(predictions)
    PRED.to_csv(OUT/'06_exploratory_heldout_cell_predictions.csv',index=False)
    PL=PRED.groupby(['fold','model','site','plot']).agg(
        n=('y','size'),reference_cover=('y','mean'),
        predicted_cover=('pred','mean')).reset_index()
    PL['abs_error_pp']=(PL.reference_cover-PL.predicted_cover).abs()*100
    PL.to_csv(OUT/'06_exploratory_sparse_plot_cover.csv',index=False)
    display(RES)
    display(PL.groupby(['model','site']).agg(
        plots=('plot','size'),MAE_pp=('abs_error_pp','mean')).round(2))
    print('EXPLORATORY ONLY: compare paired held-out cells and original plot counts;')
    print('no publication-ready full-plot cover or controlled U-Net modality test yet.')
